In [31]:
import pandas as pd
import numpy as np

from scipy import stats

In [32]:
df = pd.read_csv(
    r"D:\Election Data Project\data\processed\Elections_Feature_Engineered.csv"
)

In [34]:
numeric_columns = [

    "match_count",

    "agreement_score",

    "years_since_previous_election"

]

df[numeric_columns].describe()

,match_count,agreement_score,years_since_previous_election
count,57.000000,57.000000,51.000000
mean,2.333333,0.777778,4.607843
std,0.763763,0.254588,0.634931
min,0.000000,0.000000,4.000000
25%,2.000000,0.666667,4.000000
50%,2.000000,0.666667,5.000000
75%,3.000000,1.000000,5.000000
max,3.000000,1.000000,6.000000


In [35]:
df[numeric_columns].agg(
[
"mean",
"median",
"std",
"min",
"max"
]
)

,match_count,agreement_score,years_since_previous_election
mean,2.333333,0.777778,4.607843
median,2.000000,0.666667,5.000000
std,0.763763,0.254588,0.634931
min,0.000000,0.000000,4.000000
max,3.000000,1.000000,6.000000


### Overall Prediction Accuracy

In [36]:
correct_predictions = df["prediction_correct"].sum()

total_predictions = len(df)

accuracy = (
    correct_predictions /
    total_predictions
) * 100

print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 89.47%


In [37]:
summary = pd.DataFrame({

"Correct":[correct_predictions],

"Incorrect":[
total_predictions-correct_predictions
],

"Accuracy":[accuracy]

})

summary

,Correct,Incorrect,Accuracy
0,51,6,89.473684


### Country Statistics

In [38]:
country_statistics = (

df.groupby("Country")

.agg(

Total_Elections=("Country","count"),

Correct=("prediction_correct","sum"),

Accuracy=("prediction_correct","mean"),

Mean_Agreement=("agreement_score","mean"),

Average_Match=("match_count","mean")

)

)

country_statistics["Accuracy"]*=100

country_statistics

,Total_Elections,Correct,Accuracy,Mean_Agreement,Average_Match
Country,,,,,
Brazil,9,8,88.888889,0.777778,2.333333
Indonesia,5,4,80.000000,0.733333,2.200000
Portugal,10,10,100.000000,0.900000,2.700000
South Korea,8,7,87.500000,0.708333,2.125000
Sri Lanka,9,7,77.777778,0.740741,2.222222
United States,16,15,93.750000,0.770833,2.312500


### Country Ranking

In [39]:
country_statistics.sort_values(
"Accuracy",
ascending=False
)

,Total_Elections,Correct,Accuracy,Mean_Agreement,Average_Match
Country,,,,,
Portugal,10,10,100.000000,0.900000,2.700000
United States,16,15,93.750000,0.770833,2.312500
Brazil,9,8,88.888889,0.777778,2.333333
South Korea,8,7,87.500000,0.708333,2.125000
Indonesia,5,4,80.000000,0.733333,2.200000
Sri Lanka,9,7,77.777778,0.740741,2.222222


### Developed vs Developing

In [40]:
development_statistics=(

df.groupby("country_type")

.agg(

Total=("Country","count"),

Accuracy=("prediction_correct","mean"),

Agreement=("agreement_score","mean"),

Average_Match=("match_count","mean")

)

)

development_statistics["Accuracy"]*=100

development_statistics

,Total,Accuracy,Agreement,Average_Match
country_type,,,,
Developed,34,94.117647,0.794118,2.382353
Developing,23,82.608696,0.753623,2.260870


### Agreement Statistics

In [41]:
agreement_statistics = (

df.groupby("agreement_score")

.agg(

Total=("Country","count"),

Correct=("prediction_correct","sum")

)

)

agreement_statistics

,Total,Correct
agreement_score,,
0.000000,2,0
0.333333,4,0
0.666667,24,24
1.000000,27,27


In [42]:
agreement_statistics["Accuracy"] = (
    agreement_statistics["Correct"] /
    agreement_statistics["Total"]
) * 100

agreement_statistics

,Total,Correct,Accuracy
agreement_score,,,
0.000000,2,0,0.0
0.333333,4,0,0.0
0.666667,24,24,100.0
1.000000,27,27,100.0


### Crosstab

In [43]:
pd.crosstab(

df["agreement_score"],

df["prediction_correct"],

margins=True

)

prediction_correct,False,True,All
agreement_score,,,
0.0,2,0,2
0.333333,4,0,4
0.666667,0,24,24
1.0,0,27,27
All,6,51,57


### Confidence Interval

In [44]:
confidence = stats.binomtest(

correct_predictions,

total_predictions

)

confidence.proportion_ci()

ConfidenceInterval(low=0.7848363951977577, high=0.9603784641499192)

### Correlation Matrix

In [45]:
numeric = df.select_dtypes(
include=np.number
)

numeric.corr()

,election_year,match_count,agreement_score,regional_winner_diversity,election_decade,years_since_previous_election,country_total_elections,country_type_encoded,majority_winner_region_count
election_year,1.000000,-0.208363,-0.208363,0.150696,0.983824,0.075911,-0.363393,-0.291010,-0.150696
match_count,-0.208363,1.000000,1.000000,-0.618081,-0.210503,0.026527,0.036772,0.078728,0.618081
agreement_score,-0.208363,1.000000,1.000000,-0.618081,-0.210503,0.026527,0.036772,0.078728,0.618081
regional_winner_diversity,0.150696,-0.618081,-0.618081,1.000000,0.163479,-0.349189,0.086714,0.164395,-1.000000
election_decade,0.983824,-0.210503,-0.210503,0.163479,1.000000,0.051709,-0.326772,-0.246101,-0.163479
years_since_previous_election,0.075911,0.026527,0.026527,-0.349189,0.051709,1.000000,-0.604331,-0.181625,0.349189
country_total_elections,-0.363393,0.036772,0.036772,0.086714,-0.326772,-0.604331,1.000000,0.580062,-0.086714
country_type_encoded,-0.291010,0.078728,0.078728,0.164395,-0.246101,-0.181625,0.580062,1.000000,-0.164395
majority_winner_region_count,-0.150696,0.618081,0.618081,-1.000000,-0.163479,0.349189,-0.086714,-0.164395,1.000000


### Pearson Correlation

In [46]:
stats.pearsonr(

df["agreement_score"],

df["match_count"]

)

PearsonRResult(statistic=np.float64(0.9999999999999999), pvalue=np.float64(0.0))

### Country Variability

In [47]:
country_variation=(

df.groupby("Country")

["agreement_score"]

.std()

)

country_variation

Country
Brazil           0.235702
Indonesia        0.278887
Portugal         0.161015
South Korea      0.213623
Sri Lanka        0.433903
United States    0.200693
Name: agreement_score, dtype: float64

### Summary Table

In [48]:
summary_table=pd.DataFrame({

"Metric":[

"Total Elections",

"Correct Predictions",

"Overall Accuracy"

],

"Value":[

len(df),

correct_predictions,

accuracy

]

})

summary_table

,Metric,Value
0,Total Elections,57.000000
1,Correct Predictions,51.000000
2,Overall Accuracy,89.473684


In [49]:
country_statistics.to_csv(

r"D:\Election Data Project\data\processed\country_statistics.csv"

)

In [50]:
development_statistics.to_csv(

r"D:\Election Data Project\data\processed\development_statistics.csv"

)

In [51]:
agreement_statistics.to_csv(

r"D:\Election Data Project\data\processed\agreement_statistics.csv"

)